# Eurobot 2026 POC Results Overview

This notebook provides a quick view of simulation results: summary, trajectories, score progression, action timeline, and match animation.

In [ ]:
from pathlib import Path
import sys

from IPython.display import HTML

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from poc.metrics import summarize_batch
from poc.scenarios import build_scenario
from poc.simulator import Simulator, load_result, save_result
from poc.visualize import animate_match_overview, plot_match_overview, save_animation_media

In [ ]:
scenario_name = "baseline"
scenario = build_scenario(scenario_name, seed=7)

simulator = Simulator(
    state=scenario.game_state,
    scenario_name=scenario.name,
    opponent_policy=scenario.opponent_policy,
    dt=0.5,
)
result = simulator.run()
result.summary

In [ ]:
plot_match_overview(result);

## Animation

The animation below shows robot motion and the current time marker on the score and timeline plots.

In [ ]:
import matplotlib as mpl

SAVE_ANIMATION = False
ANIMATION_FORMAT = "mp4"  # "gif", "html", or "mp4"
ANIMATION_STEM = ROOT / "runs" / "notebook_animation"
ANIMATION_FPS = 4
ANIMATION_EMBED_LIMIT_MB = 40
mpl.rcParams["animation.embed_limit"] = ANIMATION_EMBED_LIMIT_MB

anim = animate_match_overview(
    result,
    interval=80,
    frame_stride=2,
)

if SAVE_ANIMATION:
    output_path = ANIMATION_STEM.with_suffix(f".{ANIMATION_FORMAT}")
    saved_path = save_animation_media(anim, output_path, fps=ANIMATION_FPS)
    print(f"Saved animation to {saved_path}")

HTML(anim.to_jshtml())

In [ ]:
result.planner_debug[:2]

In [ ]:
output_path = ROOT / "runs" / "notebook_demo.json"
save_result(result, output_path)
loaded = load_result(output_path)
loaded["summary"]

In [ ]:
batch_results = []
for seed in range(1, 6):
    scenario = build_scenario("baseline", seed=seed)
    simulator = Simulator(
        state=scenario.game_state,
        scenario_name=scenario.name,
        opponent_policy=scenario.opponent_policy,
        dt=0.5,
    )
    batch_results.append(simulator.run())

summarize_batch(batch_results)